# 🧪 Thực Nghiệm 2: Khảo Sát Hệ Số Lipschitz & Gradient Cho Adversarially Vulnerable Reward (ResNet-50)
### **Vanilla Guidance vs. RS-LiDAR (Randomized Smoothing)** trên GPU T4 / V100 / A100

Notebook này được thiết kế độc lập để kiểm chứng **Định lý Giảm Lipschitz (Theorem 1)** và **Hiệu Ứng Triệt Tiêu Gai Nhọn Đối Kháng (Adversarial Spike Suppression)** trên hàm phần thưởng liên tục có hằng số Lipschitz cục bộ khổng lồ ($L_{\text{local}} \gg 1$).

### 🔬 Bối cảnh & Cơ sở Khoa học:
1. **Mô hình phần thưởng**: Sử dụng **ResNet-50** tiền huấn luyện trên ImageNet-1k làm hàm phần thưởng $R(x) = f_{c^*}(x)$ (Logit của lớp mục tiêu $c^*$).
2. **Hiện tượng Đối kháng (Goodfellow et al., ICLR 2015)**: Mặc dù hàm $R(x)$ liên tục và khả vi, nhưng các mạng nơ-ron sâu cực kỳ nhạy cảm với các nhiễu tần số cao không thể cảm nhận bằng mắt thường. Điều này dẫn đến sự tồn tại của các **gai nhọn nhân tạo (Adversarial Spikes)** với độ dốc cục bộ khổng lồ ($L_{\text{local}} \gg 1$).
3. **Hệ quả đối với Diffusion Guidance**:
   - **Vanilla Guidance**: Dễ bị bẫy vào các cực tiểu cục bộ giả tạo và gradient bị định hướng sai (Gradient Misdirection).
   - **RS-LiDAR**: Làm mịn landscape bằng tích chập Gaussian Monte Carlo ($R_{\sigma_2}(x) = \mathbb{E}_{\epsilon \sim \mathcal{N}(0, \sigma_2^2 I)} [R(x + \epsilon)]$), dập tắt hoàn toàn các gai nhọn, ép gradient norm về cận lý thuyết $B / \sigma_2$.
4. **Quy mô thực nghiệm Chuẩn mực**:
   - **50 Prompts** đại diện cho **10 Super-categories** đa dạng của ImageNet (Canine, Feline, Ursidae, Avian, Insect, Vehicle, Marine, Food, Artifact, Structural).
   - **10 Hạt (Particles)** trên mỗi prompt $\rightarrow$ **500 cặp ảnh (Clean - Perturbed)**.
   - Quét toàn diện tham số: $\sigma_1 \in \{0.001, 0.005, 0.01, 0.02, 0.05\}$, $\sigma_2 \in \{0.0, 0.1, 0.25, 0.5, 1.0\}$, $M \in \{1, 2, 4, 8\}$ với kỹ thuật **Common Random Numbers (CRN)** triệt tiêu phương sai Monte Carlo.
5. **Xuất báo cáo tự động**: Bảng Benchmark Markdown, Bảng Breakdown 10 Lớp, Bảng Quét Bán Kính $\sigma_1 \to 0$, Bộ đồ thị 6-panel, và Đồ thị Lát cắt 1D Landscape Slicing (Latent Manifold & FGSM Perturbation).

## 1. Kiểm Tra Phần Cứng GPU & Cấu Hình Bộ Nhớ
Đảm bảo phiên chạy được gán GPU (T4, V100, hoặc A100).

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU hoặc A100 GPU.")

# Chống phân mảnh bộ nhớ VRAM cho mô hình Diffusion
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi

## 2. Đồng Bộ Mã Nguồn Repo & Cài Đặt Môi Trường

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, urllib.request

# 1. Đồng bộ repo RS-LiDAR từ GitHub
REPO_DIR = "/content/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện thiết yếu
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q torchvision matplotlib tqdm scipy seaborn pandas tabulate

# 3. Tiện ích chạy song song đa GPU (nếu môi trường có 2 GPU)
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    rc0 = p0.wait()
    rc1 = p1.wait()
    if rc0 != 0 or rc1 != 0:
        raise RuntimeError(f"Tiến trình GPU thất bại (GPU 0: {rc0}, GPU 1: {rc1})")

print("\n✅ Môi trường thực nghiệm Adversarially Vulnerable Reward đã sẵn sàng 100%!")

## 3. 📁 Gắn Kết Google Drive Cá Nhân (Tự Động Lưu Kết Quả Chống Mất Dữ Liệu)

In [ ]:
# @title 📁 Gắn kết Google Drive & Thiết lập thư mục lưu trữ
import os
from google.colab import drive

try:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/RS-LiDAR/results/New_Rewards'
    print(f"\n✅ Đã gắn kết Google Drive thành công!")
except Exception as e:
    print(f"⚠️ Không thể mount Google Drive: {e}. Kết quả sẽ lưu cục bộ.")
    BASE_DIR = "/content/RS-LiDAR/results/New_Rewards"

os.makedirs(BASE_DIR, exist_ok=True)
print(f"📁 Thư mục cơ sở lưu trữ: {BASE_DIR}")


## 4. Cấu Hình Tham Số Thực Nghiệm (Adversarially Vulnerable Reward)
Các tham số định cấu hình cho script đo đạc `test_adversarial_reward.py`.

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH THAM SỐ (TÙY CHỈNH TẠI ĐÂY)
# ==============================================================================
NUM_PARTICLES = 10                  # K = 10 hạt (tương đương 500 cặp ảnh với 50 prompts)
SIGMA1 = 0.05                       # Bán kính vi nhiễu chính khảo sát độ dốc secant
SIGMA1_SWEEP = "0.001,0.005,0.01,0.02,0.05" # Dải quét sigma1 khảo sát hành vi tại r -> 0
PERTURBATION_TYPE = "gaussian"      # Kiểu biến dạng vi mô: "gaussian"
SIGMA2 = 0.5                        # Bán kính làm mịn RS-LiDAR chuẩn (Primary Benchmark)
SIGMA2_SWEEP = "0.0,0.1,0.25,0.5,1.0" # Dải quét bán kính làm mịn sigma2
M_SWEEP = "1,2,4,8"                 # Dải quét số mẫu Monte Carlo M
PRIMARY_M = 4                       # Mức M chính làm mốc đánh giá
REWARD_MODE = "logit"               # Hàm thưởng: "logit" (nhạy bén), "softmax", hoặc "margin"
USE_CRN = True                      # Common Random Numbers (CRN) triệt tiêu phương sai Monte Carlo
RUN_CASE_STUDIES = True             # Tự động xuất Lát cắt 1D Landscape Slicing
OVERWRITE = True                    # Đặt True để làm mới checkpoint, chạy lại từ đầu
# ==============================================================================

import os
from datetime import datetime

# Lấy thời gian hiện tại và tạo tên thư mục động
current_time = datetime.now().strftime("%H%M_%d%m%Y")
config_str = f"K{NUM_PARTICLES}_S1{SIGMA1}_{PERTURBATION_TYPE}_S2{SIGMA2}_Msweep{M_SWEEP.replace(',', '_')}_CRN{USE_CRN}_{REWARD_MODE}"
task_name = "Adversarial_Rewards"
folder_name = f"{task_name}_{config_str}_{current_time}"

# Cập nhật lại TEST_OUTPUT_DIR bao gồm cấu hình và thời gian
TEST_OUTPUT_DIR = os.path.join(BASE_DIR, folder_name)
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

print(f"📊 Số lượng: {NUM_PARTICLES} hạt/prompt (Tổng cộng {NUM_PARTICLES * 50} cặp mẫu ảnh)")
print(f"🔬 Nhiễu biến dạng sigma1 = {SIGMA1} (sweep: {{{SIGMA1_SWEEP}}}) | Bán kính làm mịn chuẩn sigma2 = {SIGMA2}")
print(f"📈 Dải khảo sát sigma2: {{{SIGMA2_SWEEP}}} | Dải quét Monte Carlo M: {{{M_SWEEP}}} (Primary M={PRIMARY_M})")
print(f"🎯 Reward Mode: '{REWARD_MODE}' | CRN: {USE_CRN} | Case Studies: {RUN_CASE_STUDIES}")
print(f"📁 Thư mục lưu kết quả: {TEST_OUTPUT_DIR}")


## 5. Thực Thi Đo Đạc Lipschitz Song Song (Tự động tải Model & Run)
Script `test_adversarial_reward.py` sẽ nạp Stable Diffusion 1.5 và ResNet-50, tự động sinh ảnh sạch và ảnh biến dạng vi mô, đo đạc độ dốc secant, tính Autograd gradient norm $\|\nabla_x R(x)\|$, và lưu dữ liệu phân tích.

In [ ]:
# 1. Dọn dẹp tiến trình cũ & giải phóng bộ nhớ
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
!pkill -f "test_adversarial_reward.py" || true

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
ov_flag = "--overwrite" if OVERWRITE else ""
crn_flag = "--use_crn" if USE_CRN else "--no_crn"
cs_flag = "--run_case_studies" if RUN_CASE_STUDIES else ""

print(f"\n🚀 Bắt đầu đo Lipschitz trên Adversarially Vulnerable Reward (ResNet-50) với {n_gpus} GPU...\n")

if n_gpus >= 2:
    print("⚡ [2 GPU] Đang chạy song song trên GPU 0 (Shard 0) và GPU 1 (Shard 1)...")
    cmd0 = f"""python -u test_adversarial_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --sigma1_list "{SIGMA1_SWEEP}" \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --m_list "{M_SWEEP}" \
        --primary_m {PRIMARY_M} \
        --reward_mode {REWARD_MODE} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 0 --gpu_id 0 {cs_flag} {crn_flag} {ov_flag}"""

    cmd1 = f"""python -u test_adversarial_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --sigma1_list "{SIGMA1_SWEEP}" \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --m_list "{M_SWEEP}" \
        --primary_m {PRIMARY_M} \
        --reward_mode {REWARD_MODE} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 1 --gpu_id 1 {crn_flag} {ov_flag}"""
    run_commands_parallel(cmd0, cmd1)
else:
    print("⚡ [1 GPU] Đang chạy đo trên 1 GPU...")
    !python -u test_adversarial_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --sigma1_list "{SIGMA1_SWEEP}" \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --m_list "{M_SWEEP}" \
        --primary_m {PRIMARY_M} \
        --reward_mode {REWARD_MODE} \
        --output_dir "{TEST_OUTPUT_DIR}" {cs_flag} {crn_flag} {ov_flag}

print("\n✅ HOÀN THÀNH ĐO ĐẠC! Tiến hành gộp shard và tạo biểu đồ...")

## 6. Tự Động Gộp Kết Quả Shards & Đồ Thị Hóa
Khâu này sẽ kết hợp các checkpoint shard, tính toán ma trận thống kê, đối chiếu cận lý thuyết Theorem 1, và vẽ bộ biểu đồ khoa học chất lượng cao.

In [ ]:
print("🔄 Đang gộp kết quả, xuất bảng Markdown và vẽ biểu đồ tổng hợp...")
!python -u test_adversarial_reward.py --merge_shards --output_dir "{TEST_OUTPUT_DIR}" --sigma2 {SIGMA2} --primary_m {PRIMARY_M} --sigma1 {SIGMA1}
print("✅ Hoàn tất gộp dữ liệu và tạo báo cáo!")

## 7. Hiển Thị Báo Cáo Khoa Học & Trực Quan Hóa Chi Tiết
Hiển thị các Bảng Markdown tổng hợp cùng toàn bộ các đồ thị phân tích đa chiều (bao gồm sweep bán kính vi nhiễu $\sigma_1 \to 0$).

In [ ]:
from IPython.display import Image, display, Markdown
import os, glob

# 1. Hiển thị Bảng 1 & Bảng 3: Benchmark Chính & Quét sigma1
tbl1_path = os.path.join(TEST_OUTPUT_DIR, "adversarial_lipschitz_benchmark_table.md")
if os.path.exists(tbl1_path):
    print("\n📋 BẢNG 1 & BẢNG 3: LIPSCHITZ & GRADIENT BENCHMARK CHO ADVERSARIALLY VULNERABLE REWARD")
    with open(tbl1_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

# 2. Hiển thị Bảng 2: Breakdown 10 ImageNet Super-categories
tbl2_path = os.path.join(TEST_OUTPUT_DIR, "adversarial_class_breakdown_table.md")
if os.path.exists(tbl2_path):
    print("\n📋 BẢNG 2: PHÂN TÍCH CHI TIẾT 10 IMAGENET SUPER-CATEGORIES")
    with open(tbl2_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

# 3. Hiển thị Dashboard 6-Panel toàn diện
dash_path = os.path.join(TEST_OUTPUT_DIR, "adversarial_lipschitz_combined_dashboard.png")
if os.path.exists(dash_path):
    print(f"\n📊 DASHBOARD TỔNG HỢP 6 PANEL: {os.path.basename(dash_path)}")
    display(Image(filename=dash_path))

# 4. Hiển thị Case Study: 1D Landscape Slicing 2x2
slice_path = os.path.join(TEST_OUTPUT_DIR, "adversarial_1d_landscape_slice.png")
if os.path.exists(slice_path):
    print(f"\n🏔️ CASE STUDY 2 - 1D LANDSCAPE INTERPOLATION SLICE: {os.path.basename(slice_path)}")
    display(Image(filename=slice_path))

# 5. Hiển thị Đồ thị Quét Bán Kính Vi Nhiễu sigma1 Ablation (Spike Explosion vs Bounded)
sigma1_plot = os.path.join(TEST_OUTPUT_DIR, "adversarial_sigma1_ablation.png")
if os.path.exists(sigma1_plot):
    print(f"\n📈 ĐỒ THỊ QUÉT SIGMA1 ABLATION: {os.path.basename(sigma1_plot)}")
    display(Image(filename=sigma1_plot))

# 6. Hiển thị các bảng đồ thị thành phần khác
other_plots = [
    "adversarial_table_benchmark.png",
    "adversarial_table_categories_breakdown.png",
    "adversarial_table_sigma1_sweep.png",
    "adversarial_sigma_ablation.png",
    "adversarial_gradient_norm_dist.png"
]
for p_name in other_plots:
    p_path = os.path.join(TEST_OUTPUT_DIR, p_name)
    if os.path.exists(p_path):
        print(f"\n📈 Đồ thị: {p_name}")
        display(Image(filename=p_path))


## 8. Đóng Gói Toàn Bộ Kết Quả (Tự Động Nén Zip Lưu Vào Google Drive)

In [ ]:
import os
zip_name = os.path.basename(TEST_OUTPUT_DIR) + ".zip"
# Ghi file zip ra thư mục cha của TEST_OUTPUT_DIR
zip_path = os.path.join(os.path.dirname(TEST_OUTPUT_DIR), zip_name)
print(f"📦 Đang nén toàn bộ kết quả vào {zip_path}...")

parent_dir = os.path.dirname(TEST_OUTPUT_DIR)
target_folder = os.path.basename(TEST_OUTPUT_DIR)
!cd "{parent_dir}" && zip -r -q "{zip_name}" "{target_folder}"

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"👉 Bạn có thể tải file này từ tab Files bên trái Colab, hoặc vào Drive '{zip_path}' để lấy.")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")
